In [12]:
# importing Libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
import os





In [13]:
# Loading the data sets
df_movies = pd.read_csv('../data/movies.csv') 
df_ratings = pd.read_csv('../data/ratings.csv')

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [14]:
#  genre filtering
def movies_by_genres(genre):
    return df_movies[df_movies["genres"].str.contains(genre, case=False)]["title"]

movies_by_genres("comedy")

0                                Toy Story (1995)
2                         Grumpier Old Men (1995)
3                        Waiting to Exhale (1995)
4              Father of the Bride Part II (1995)
6                                  Sabrina (1995)
                          ...                    
9732                    Gintama: The Movie (2010)
9734                          Silver Spoon (2014)
9737    Black Butler: Book of the Atlantic (2017)
9738                 No Game No Life: Zero (2017)
9741          Andrew Dice Clay: Dice Rules (1991)
Name: title, Length: 3756, dtype: object

In [15]:
import requests

# Functions

def get_poster(title):

    clean_title = title.split("(")[0].strip()
    year = title.split("(")[1].replace(")","").strip()

    url = "https://api.themoviedb.org/3/search/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "query": clean_title,
        "year": year
    }
    
    response = requests.get(url, params=params)
    data = response.json()

    if not data["results"]:                       # handling edge cases  if the result is empty 
        return None
    else :
        poster_path = data['results'][0]['poster_path']

        full_url =  "https://image.tmdb.org/t/p/w500" + poster_path
        
        return full_url
        




In [16]:
# Content based filtering using TF-IDF
df_movies['features'] = df_movies['title'] + " " + df_movies['genres'].str.replace("|", " ")

df_movies.head(3)

,movieId,title,genres,features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995) Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy Romance


In [17]:
def get_movie_description(title):
    try:
        clean_title = title.split("(")[0].strip()
        year = title.split("(")[1].replace(")","").strip()
        
        url = "https://api.themoviedb.org/3/search/movie"
        params = {
            "api_key":"1b4407af1aa4455ef0bcded4e2a7edc6",
            "query":  clean_title,
            "year":  year
        }
        
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        
        if not data["results"]:
            return ""
            
        return data["results"][0].get('overview', "")
        
    except:
        return ""

print(get_movie_description("Toy Story (1995)"))


Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.


In [18]:
avg_ratings = df_ratings.groupby('movieId')['rating'].mean()
print(avg_ratings.head())
print(f"Total movies with ratings: {len(avg_ratings)}")

movieId
1    3.920930
2    3.431818
3    3.259615
4    2.357143
5    3.071429
Name: rating, dtype: float64
Total movies with ratings: 9724


In [19]:
def get_cb_recommendations(genre, n=5):
    
    genre_movies = movies_by_genres(genre)
    
    if len(genre_movies) == 0:
        return []
    
    sample = genre_movies.sample(min(10, len(genre_movies))).tolist()
    
    features = []
    titles = []
    movie_ids = []
    
    # Genre reference
    genre_reference = " ".join([genre] * 5)
    features.append(genre_reference)
    titles.append("REFERENCE")
    movie_ids.append(None)
    
    # Movie features
    for title in sample:
        desc = get_movie_description(title)
        genre_text = df_movies[df_movies['title'] == title]['genres'].values[0].replace("|", " ")
        
        mid = df_movies[df_movies['title'] == title]['movieId'].values[0]
        
        combined = f"{genre_text} {desc}"
        features.append(combined)
        titles.append(title)
        movie_ids.append(mid)
    
    # TF-IDF
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(features)
    
    # Cosine Similarity
    sim_matrix = cosine_similarity(tfidf_matrix)
    
    # Similarity scores vs reference
    sim_scores = list(enumerate(sim_matrix[0]))
    sim_scores = sim_scores[1:]  # Skip reference
    
    # Multiply similarity × avg rating 🎯
    weighted_scores = []
    for idx, sim in sim_scores:
        mid = movie_ids[idx]
        avg_rating = avg_ratings.get(mid, 3.0)
        weighted = sim * avg_rating
        weighted_scores.append((idx, weighted))
    
    # Sort by weighted score
    weighted_scores = sorted(
        weighted_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Return top n
    recommended = [
        titles[i[0]] 
        for i in weighted_scores[:n]
    ]
    return recommended

In [20]:
print(get_cb_recommendations("Comedy"))

['Blue Mountain State: The Rise of Thadland (2015)', 'Animals are Beautiful People (1974)', 'Trailer Park Boys (1999)', '27 Dresses (2008)', "Butcher's Wife, The (1991)"]


In [21]:
genre_movies = movies_by_genres("Comedy")
print(len(genre_movies))
print(genre_movies.head())

3756
0                      Toy Story (1995)
2               Grumpier Old Men (1995)
3              Waiting to Exhale (1995)
4    Father of the Bride Part II (1995)
6                        Sabrina (1995)
Name: title, dtype: object


In [22]:
print(get_cb_recommendations("Comedy"))
print(get_cb_recommendations("Comedy"))

['Eddie Murphy Delirious (1983)', 'Sorry to Bother You (2018)', 'Cyrano de Bergerac (1990)', "Operation 'Y' & Other Shurik's Adventures (1965)", 'Dedication (2007)']
['Go (1999)', 'Playing for Keeps (2012)', 'Cyrano de Bergerac (1990)', 'I Served the King of England (Obsluhoval jsem anglického krále) (2006)', 'Secondhand Lions (2003)']
